In [54]:
import numpy as np
import pandas as pd


In [55]:
train = pd.read_csv("D:/Prithiv/AIML PROJECT/DATASET/train.csv")
dev = pd.read_csv("D:/Prithiv/AIML PROJECT/DATASET/dev.csv")
test = pd.read_csv("D:/Prithiv/AIML PROJECT/DATASET/test.csv")

In [56]:
print(train["label"].value_counts(normalize=True)*100)


label
0    61.955733
1    38.044267
Name: proportion, dtype: float64


In [57]:
print(train['text'].duplicated().sum())
print(test['text'].duplicated().sum())
print(dev['text'].duplicated().sum())

1039
92
20


In [58]:
train_texts = set(train["text"].astype(str))
dev_texts = set(dev["text"].astype(str))
overlap_text=train_texts & dev_texts
print(len(overlap_text))
test_texts = set(test["text"].astype(str))
overlap_train_test = train_texts & test_texts
print("Train-Test overlapping texts:", len(overlap_train_test))
overlap_dev_test = dev_texts & test_texts
print("Dev-Test overlapping texts:", len(overlap_dev_test))

292
Train-Test overlapping texts: 548
Dev-Test overlapping texts: 74


In [59]:
for d in (train, dev, test):
    d["key"] = d["text"].str.lower().str.replace(r"\s+", "", regex=True)

In [60]:
conflict = train.groupby("key")["label"].transform("nunique") > 1
print("conflict rows removed:", conflict.sum())
train = train[~conflict].drop_duplicates(subset="key")

dev_full = dev.copy()
dev = dev.drop_duplicates(subset="key")
dev = dev[~dev["key"].isin(train["key"])]

print("train:", len(train), "| dev:", len(dev), "| test:", len(test))
print(train["label"].value_counts(normalize=True) * 100)
print(dev["label"].value_counts(normalize=True) * 100)

conflict rows removed: 34
train: 4468 | dev: 475 | test: 1576
label
0    63.943599
1    36.056401
Name: proportion, dtype: float64
label
0    68.631579
1    31.368421
Name: proportion, dtype: float64


In [61]:
print(train['text'].isnull().sum())
print(test['text'].isnull().sum())
print(dev['text'].isnull().sum())
print(test['text'].duplicated().sum())
print(train['text'].duplicated().sum())
print(dev['text'].duplicated().sum())


0
0
0
92
0
0


In [62]:
tr_k, dv_k, te_k = set(train["key"]), set(dev["key"]), set(test["key"])
print("train & dev :", len(tr_k & dv_k))
print("train & test:", len(tr_k & te_k))
print("dev & test  :", len(dv_k & te_k))

train & dev : 0
train & test: 546
dev & test  : 72


In [63]:
for d in (train,test,dev):
    d['text_clean']=(d['text'] .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
                 .str.replace(r"@[\w\-.]*\w", " ", regex=True)
                 .str.replace("\u200b", "", regex=False)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip())

In [64]:
for name, d in [("train", train), ("dev", dev), ("test", test)]:
    print(name, len(d), "| missing:", d["text_clean"].isnull().sum(),
          "| empty:", (d["text_clean"] == "").sum(), "| duplicates:", d["key"].duplicated().sum())

print(train["label"].value_counts(normalize=True) * 100)
print(dev["label"].value_counts(normalize=True) * 100)

train 4468 | missing: 0 | empty: 0 | duplicates: 0
dev 475 | missing: 0 | empty: 0 | duplicates: 0
test 1576 | missing: 0 | empty: 0 | duplicates: 92
label
0    63.943599
1    36.056401
Name: proportion, dtype: float64
label
0    68.631579
1    31.368421
Name: proportion, dtype: float64


In [66]:
path="D:\Prithiv\AIML PROJECT\DATASET"
train[["id", "text", "label"]].to_csv(path + "train_clean.csv", index=False)
dev[["id", "text", "label"]].to_csv(path + "dev_clean.csv", index=False)
test[["id", "text"]].to_csv(path + "test_clean.csv", index=False)

In [ ]:
for name, d in [("train", train), ("dev", dev), ("test", test)]:
    print(name,
          "| @user:", d["text_clean"].str.contains("@").sum(),
          "| url:", d["text_clean"].str.contains("http|www").sum(),
          "| double space:", d["text_clean"].str.contains("  ").sum(),
          "| empty:", (d["text"] == "").sum())

train | @user: 2 | url: 6 | double space: 757 | empty: 0
dev | @user: 0 | url: 0 | double space: 89 | empty: 0
test | @user: 0 | url: 1 | double space: 261 | empty: 0
